# 06 — Build metadata tables for the training imagery

**Purpose.** Turns the raw output of the training pull (notebook 03) into four flat CSVs, one row per
scene, keyed on `file_name`. In Athena they become the `processed_logs`, `file_attributes`,
`image_attributes` and `ee_metadata` tables in `glacier-view`. `sql/training_data_query.sql` filters
them to the one image per glacier used for training, and the `identify_inference_*` queries read them
to choose the inference glaciers.

1. `processed_logs.csv`: parsed from the pull log (pull date, `glims_id`, CRS, cloud cover,
   scene time, image quality).
2. `file_attributes.csv`: from each file's name and size (date, satellite, bytes).
3. `image_attributes.csv`: from opening each GeoTIFF (shape, bounds, band count, EPSG,
   resolution, and counts of zero, no-data and negative pixels). Unreadable files get no row.
4. `ee_metadata.csv`: Earth Engine's metadata, flattened; the first record is kept when Earth
   Engine returns more than one per scene.

The code is in `glacierview.metadata`. Athena reads these CSVs by column position, so the column
order there is part of the contract.

The training `ee_metadata.csv` carries a `version` column after `file_name`; the inference table
doesn't.

## Inputs
All under `config.landing_zone(config.TRAINING_DATA_LABEL)` (gitignored):
- **`landsat/<glims_id>/*.tif`** and **`meta_data/metadata_list_l{5,7,8}`**: from notebook 03. 18,093
  glacier folders locally; the S3 copy is `s3://raw-training-images-t1-l2-sr` (us-west-1). Every file
  is read in full, so this is slow.
- **`logs/training_log_1.log`**: the original pull's log. If it's missing, the notebook uses
  `log_full_training_data.log`, which a fresh run of notebook 03 writes.

## Outputs
- The four CSVs in `config.processed_metadata_dir(config.TRAINING_DATA_LABEL)`. Gitignored.

## Manual steps afterwards (not in code)
1. Upload each CSV to its prefix in `s3://training-images-metadata-t1-l2-sr`
   (`{processed_logs,file_attributes,image_attributes,ee_metadata}/`); the Glue tables point there.
2. Run `sql/training_data_query.sql` in Athena and save the result as
   `filtered_training_data_summer_months.csv` in the same directory. Notebook 10 reads it.

In [ ]:
from glacierview import config, log
from glacierview import metadata as md

log.configure()
LABEL = config.TRAINING_DATA_LABEL
landsat = config.landsat_dir(LABEL)
out = config.processed_metadata_dir(LABEL)
out.mkdir(parents=True, exist_ok=True)
glims_ids = md.glacier_ids(landsat)

In [ ]:
log_path = config.log_dir(LABEL) / "training_log_1.log"
if not log_path.exists():
    log_path = config.log_dir(LABEL) / "log_full_training_data.log"
md.parse_pull_log(log_path).to_csv(out / "processed_logs.csv", index=False)

In [ ]:
files, images = md.scan_scenes(landsat, glims_ids)
files.to_csv(out / "file_attributes.csv", index=False)
images.to_csv(out / "image_attributes.csv", index=False)

In [ ]:
ee_metadata = md.read_ee_metadata(landsat, glims_ids, include_version=True)
ee_metadata.to_csv(out / "ee_metadata.csv", index=False)